# 1. Me importo todas las librerías que voy a necesitar

In [1]:
import pandas as pd
import seaborn as sns
import json
import numpy as np
import requests
import os
import matplotlib.pyplot as plt

# 2 Hago merge de todos los dataset limpios

In [2]:
vivienda = pd.read_csv("../data/Depurados/limpio_vivienda.csv")
turismo = pd.read_csv("../data/Depurados/limpio_turistas_pernoctan.csv")
inflacion = pd.read_csv("../data/Depurados/limpio_inflacion.csv")
pib = pd.read_csv("../data/Depurados/limpio_pib.csv")

In [3]:
print(vivienda)
print(turismo)
print(inflacion)
print(pib)

              pais iso3  anio  crec._precio_(%)
0        Australia  AUS  2015               9.0
1        Australia  AUS  2016               5.5
2        Australia  AUS  2017               8.4
3        Australia  AUS  2018              -1.4
4        Australia  AUS  2019              -4.0
..             ...  ...   ...               ...
565  United States  USA  2020               6.7
566  United States  USA  2021              15.1
567  United States  USA  2022              12.8
568  United States  USA  2023               4.0
569  United States  USA  2024               4.3

[570 rows x 4 columns]
        pais iso3  anio  numero_turistas  crec._turistas_(%)
0      Aruba  ABW  2015          1225000                 NaN
1      Aruba  ABW  2016          1102000               -10.0
2      Aruba  ABW  2017          1070500                -2.9
3      Aruba  ABW  2018          1082000                 1.1
4      Aruba  ABW  2019          1119000                 3.4
...      ...  ...   ...           

# 3 Quitamos columnas con números absolutos de turismo, pib e inflación para dejar solo tasa de crecimiento

In [4]:
turismo = turismo[["pais", "iso3", "anio", "crec._turistas_(%)"]]
inflacion = inflacion[["pais", "iso3", "anio", "inflacion_(%)"]]
pib = pib[["pais", "iso3", "anio", "crecim_pib_(%)"]]
print(turismo.shape, inflacion.shape, pib.shape)

(1518, 4) (1811, 4) (1858, 4)


In [5]:
print(vivienda)
print(turismo)
print(inflacion)
print(pib)

              pais iso3  anio  crec._precio_(%)
0        Australia  AUS  2015               9.0
1        Australia  AUS  2016               5.5
2        Australia  AUS  2017               8.4
3        Australia  AUS  2018              -1.4
4        Australia  AUS  2019              -4.0
..             ...  ...   ...               ...
565  United States  USA  2020               6.7
566  United States  USA  2021              15.1
567  United States  USA  2022              12.8
568  United States  USA  2023               4.0
569  United States  USA  2024               4.3

[570 rows x 4 columns]
        pais iso3  anio  crec._turistas_(%)
0      Aruba  ABW  2015                 NaN
1      Aruba  ABW  2016               -10.0
2      Aruba  ABW  2017                -2.9
3      Aruba  ABW  2018                 1.1
4      Aruba  ABW  2019                 3.4
...      ...  ...   ...                 ...
1513  Zambia  ZMB  2017                 5.5
1514  Zambia  ZMB  2018                 6.2
1515

# 4 Hago el merge entre todos los dataset usando el código iso3

In [6]:
datos_mergeados = vivienda.merge( turismo[["iso3", "anio", "crec._turistas_(%)"]], on=["iso3", "anio"], how="left")
print(datos_mergeados.shape)

(570, 5)


In [7]:
datos_mergeados = datos_mergeados.merge(inflacion[["iso3", "anio", "inflacion_(%)"]],on=["iso3", "anio"], how="left")
print(datos_mergeados.shape)


(570, 6)


In [8]:
datos_mergeados = datos_mergeados.merge(pib[["iso3", "anio", "crecim_pib_(%)"]],on=["iso3", "anio"], how="left")
print(datos_mergeados.shape)

(570, 7)


In [9]:
print(datos_mergeados)

              pais iso3  anio  crec._precio_(%)  crec._turistas_(%)  \
0        Australia  AUS  2015               9.0                 NaN   
1        Australia  AUS  2016               5.5                 NaN   
2        Australia  AUS  2017               8.4                 NaN   
3        Australia  AUS  2018              -1.4                 NaN   
4        Australia  AUS  2019              -4.0                 NaN   
..             ...  ...   ...               ...                 ...   
565  United States  USA  2020               6.7               -75.8   
566  United States  USA  2021              15.1                16.0   
567  United States  USA  2022              12.8               127.9   
568  United States  USA  2023               4.0                30.7   
569  United States  USA  2024               4.3                 9.1   

     inflacion_(%)  crecim_pib_(%)  
0              1.5             NaN  
1              1.3           -12.1  
2              1.9             8.1  

# 5 Compruebo los NaN que hay por columnas

In [10]:
datos_mergeados.isnull().sum()

pais                    0
iso3                    0
anio                    0
crec._precio_(%)        0
crec._turistas_(%)    162
inflacion_(%)           0
crecim_pib_(%)         57
dtype: int64

# 6 Elimino los NaN

In [11]:
datos_mergeados = datos_mergeados.dropna()
datos_mergeados.shape

(408, 7)

In [14]:
datos_mergeados = datos_mergeados.reset_index(drop=True)
print(datos_mergeados)

              pais iso3  anio  crec._precio_(%)  crec._turistas_(%)  \
0          Austria  AUT  2016               7.4                 5.2   
1          Austria  AUT  2017               3.8                 4.8   
2          Austria  AUT  2018               6.9                 4.6   
3          Austria  AUT  2019               4.0                 3.5   
4          Austria  AUT  2020               7.0               -52.7   
..             ...  ...   ...               ...                 ...   
403  United States  USA  2020               6.7               -75.8   
404  United States  USA  2021              15.1                16.0   
405  United States  USA  2022              12.8               127.9   
406  United States  USA  2023               4.0                30.7   
407  United States  USA  2024               4.3                 9.1   

     inflacion_(%)  crecim_pib_(%)  
0              0.9             2.6  
1              2.1             4.7  
2              2.0             8.5  

In [17]:
datos_mergeados.head(22)

,pais,iso3,anio,crec._precio_(%),crec._turistas_(%),inflacion_(%),crecim_pib_(%)
0,Austria,AUT,2016,7.4,5.2,0.9,2.6
1,Austria,AUT,2017,3.8,4.8,2.1,4.7
2,Austria,AUT,2018,6.9,4.6,2.0,8.5
3,Austria,AUT,2019,4.0,3.5,1.5,-2.6
4,Austria,AUT,2020,7.0,-52.7,1.4,-2.3
5,Austria,AUT,2021,11.8,-15.7,2.8,10.1
6,Austria,AUT,2022,10.4,106.0,8.5,-2.4
7,Belgium,BEL,2016,2.3,-10.5,2.0,2.3
8,Belgium,BEL,2017,3.5,12.1,2.1,5.2
9,Belgium,BEL,2018,3.0,8.8,2.1,7.8


# 7 Calculo las medias agregadas de todos los países: agrupo por años y saco media de todas las columnas

In [19]:
media_mundial = datos_mergeados.groupby("anio")[["crec._precio_(%)", "crec._turistas_(%)", "inflacion_(%)", "crecim_pib_(%)"]].mean().round(1)
print(media_mundial)

      crec._precio_(%)  crec._turistas_(%)  inflacion_(%)  crecim_pib_(%)
anio                                                                     
2016               5.2                 7.0            1.4             1.8
2017               5.8                 9.5            2.2             7.5
2018               5.4                 5.3            2.3             7.4
2019               4.8                 2.7            2.1            -0.5
2020               5.3               -72.6            1.4            -4.3
2021               9.3                 4.8            3.4            15.2
2022              11.7               441.5           10.0             1.4
2023               3.2               155.9            6.2             8.3
2024               4.4                 9.7            3.0             4.4
